<a href="https://colab.research.google.com/github/pramodkumarw/Github-Colab/blob/main/Colab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langgraph

In [ ]:
# STEP 1: SET UP THE ENVIRONMENT

# STEP 1.1 INSTALL THE REQUIRED PACKAGES
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install langchain-groq

In [6]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# from rich import print
from google.colab import userdata

GROQ_API_KEY=userdata.get("GROQ_API_KEY")
llm=ChatGroq(model="openai/gpt-oss-120b", groq_api_key=GROQ_API_KEY)


In [ ]:
!pip install langchain
!pip install langgrah


In [7]:
from pydantic import BaseModel
from typing import Literal
from langgraph.graph import StateGraph
from langchain.messages import HumanMessage, SystemMessage

class tweetState(BaseModel):
  topic:str
  tweet:str
  evaluation:Literal["approved","need_improvement"]
  feedback:str
  iteration:int
  max_iteration:int

In [ ]:
def generate_tweet(state:tweetState):
  messages=[
      SystemMessage(content="you are a funny and cleaver tweeter infulencer"),
     HumanMessage(content=f"""
          Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

          Rules:
          - Do NOT use question-answer format.
          - Max 280 characters.
          - Use observational humor, irony, sarcasm, or cultural references.
          - Think in meme logic, punchlines, or relatable takes.
          - Use simple, day to day english
          """)
  ]
  res=llm.invoke(messages).content
  return {"tweet":res}

In [ ]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

structured_evaluator_llm = llm.with_structured_output(TweetEvaluation)

def evaluate_tweet(state:tweetState):
  messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
          Evaluate the following tweet:

          Tweet: "{state['tweet']}"

          Use the criteria below to evaluate the tweet:

          1. Originality – Is this fresh, or have you seen it a hundred times before?
          2. Humor – Did it genuinely make you smile, laugh, or chuckle?
          3. Punchiness – Is it short, sharp, and scroll-stopping?
          4. Virality Potential – Would people retweet or share it?
          5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

          Auto-reject if:
          - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
          - It exceeds 280 characters
          - It reads like a traditional setup-punchline joke
          - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

          ### Respond ONLY in structured format:
          - evaluation: "approved" or "needs_improvement"
          - feedback: One paragraph explaining the strengths and weaknesses
          """)
]
  res=structured_evaluator_llm.invoke(messages).content
  return {"tweet":res}

In [ ]:
graph=StateGraph()
graph.add_node("generate_tweet",generate_tweet)
graph.add_node("evaluate_tweet",evaluate_tweet)
graph.add_node("optimize_tweet",optimize_tweet)
